# 03 — 독립성 분석 · Gate 1 (M4)

**목적:** Phase 1 세 정보원의 타이밍 독립성을 측정하고, 결합 포함 명단을 확정한다.

**공통기간:** 1990-01-31~ (realized·blend 워밍업 바닥, 6개 변형 모두 유효)  
**사전 등록 판정 임계 (코드 상단 상수 — 결과 보기 전 고정, 사후 변경 금지):**
- `|ρ_active| < 0.30` → 저상관 (기준2 포함 자격)
- `|ρ_active| > 0.50` → 고상관 (분산이득 약함)
- `0.30 ≤ |ρ_active| ≤ 0.50` → 회색지대, 신호 w_t 상관(i) 보조 병기


In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if "" not in sys.path:
    sys.path.insert(0, os.getcwd())

import warnings, yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from src.data_loader import load_all
from src.indicators.base import standalone_data
from src.indicators.volatility import VolatilityIndicator
from src.indicators.credit import CreditIndicator
from src.indicators.trend import TrendIndicator
from src.backtest import run
from src.benchmarks import buy_and_hold, equal_exposure
from src.metrics import summary, avg_exposure

print("cwd:", os.getcwd())


In [ ]:
# ★ Gate 1 사전 등록 임계 (결과 보기 전 고정 — 사후 변경 금지)
# PROJECT_PLAN 6.3 기준2: 능동수익 상관(iii) |ρ| 기반
CORR_LOW  = 0.30   # |ρ| < 0.30  → 저상관 (기준2 포함 자격)
CORR_HIGH = 0.50   # |ρ| > 0.50  → 고상관 (분산이득 약함)
# 0.30 ≤ |ρ| ≤ 0.50 → 회색지대, 신호 w_t 상관(i) 보조 근거 병기

COMMON_START = pd.Timestamp("1990-01-31")   # 공통기간 시작일 (realized·blend 워밍업 바닥)

with open('config/base.yaml') as f:
    base_cfg = yaml.safe_load(f)
with open('config/indicators/credit.yaml') as f:
    credit_yaml = yaml.safe_load(f)
cfg = {
    **base_cfg,
    'percentile_window': credit_yaml['percentile_window'],
    'theta_low_pct':     credit_yaml['map']['theta_low_pct'],
    'theta_high_pct':    credit_yaml['map']['theta_high_pct'],
}

data_raw = load_all(cfg)
print(f"CORR_LOW={CORR_LOW}, CORR_HIGH={CORR_HIGH}, COMMON_START={COMMON_START.date()}")


In [ ]:
# --- 6개 변형 정의 ---
VARIANTS = {
    'vix':      (VolatilityIndicator(), {**cfg, 'vol_estimator':'vix'}, "volatility"),
    'realized': (VolatilityIndicator(), {**cfg, 'vol_estimator':'realized','realized_lookback':21}, "volatility"),
    'blend':    (VolatilityIndicator(), {**cfg, 'vol_estimator':'blend'}, "volatility"),
    'credit':   (CreditIndicator(),    cfg, "credit"),
    'ma200':    (TrendIndicator(),     {**cfg, 'rule':'ma200','w_floor':0.0}, "trend"),
    'tsmom12':  (TrendIndicator(),     {**cfg, 'rule':'tsmom12','w_floor':0.0}, "trend"),
}
FAMILIES = {
    'vix':'vol','realized':'vol','blend':'vol',
    'credit':'credit',
    'ma200':'trend','tsmom12':'trend',
}
NAMES = list(VARIANTS.keys())

signals_common = {}
returns_strat  = {}
returns_bnh    = {}
common_meta    = {}

for name, (ind, c, src) in VARIANTS.items():
    d_src   = standalone_data(data_raw, src)
    w_full  = ind.signal(d_src, c)
    w       = w_full.loc[COMMON_START:].dropna()
    r_eq    = d_src['sp500tr'].pct_change().reindex(w.index).fillna(0.0)
    rf      = d_src['rf'].reindex(w.index)
    rf_d    = rf / 100.0 / 252.0
    res     = run(w, r_eq, rf, c)
    mw      = avg_exposure(res['weights'])
    bnh     = buy_and_hold(r_eq, rf, c)
    ee      = equal_exposure(mw, r_eq, rf, c)
    m_s     = summary(res['equity_net'], res['returns_net'], r_eq, rf_d, res['weights'], res['turnover'])
    m_e     = summary(ee['equity_net'],  ee['returns_net'],  r_eq, rf_d, ee['weights'],  ee['turnover'])
    signals_common[name] = w
    returns_strat[name]  = res['returns_net']
    returns_bnh[name]    = bnh['returns_net']
    common_meta[name]    = dict(mean_w=mw, m_strat=m_s, m_ee=m_e)
    print(f"{name:8s}: N={len(w)}, eval_start={w.index[0].date()}, mean_w={mw:.4f}")


## 1. 공통기간 재백테스트 — common1990_metrics.csv

In [ ]:
rows = []
for name in NAMES:
    d = common_meta[name]
    for label, m in [(f'strat_{name}', d['m_strat']),
                     (f"ee({d['mean_w']:.3f})", d['m_ee'])]:
        rows.append({
            'variant': name, 'family': FAMILIES[name],
            'strategy': label,
            'eval_start': str(COMMON_START.date()),
            'CAGR':        f"{m['cagr']:.2%}",
            'Vol':         f"{m['annual_vol']:.2%}",
            'Sharpe':      f"{m['sharpe']:.3f}",
            'Sortino':     f"{m['sortino']:.3f}",
            'MDD':         f"{m['mdd']:.2%}",
            'Calmar':      f"{m['calmar']:.3f}",
            'UpCap':       f"{m['up_capture']:.2%}",
            'DnCap':       f"{m['down_capture']:.2%}",
            'Turnover/yr': f"{m['annual_turnover']:.2f}",
            'AvgExp':      f"{m['avg_exposure']:.3f}",
        })

df_cmn = pd.DataFrame(rows)
df_cmn.to_csv('results/common1990_metrics.csv', index=False)
print("저장: results/common1990_metrics.csv")
print(df_cmn.to_string(index=False))


## 2. M3 max기간 vs common1990 Sharpe 비교 (추세 기간 효과)

In [ ]:
# M3 max기간 확정값 (하드코딩 — M3 CSV에서 읽은 값)
m3_max = {
    'vix':      {'strat': 0.542, 'ee': 0.522},
    'realized': {'strat': 0.607, 'ee': 0.536},
    'blend':    {'strat': 0.578, 'ee': 0.535},
    'credit':   {'strat': 0.541, 'ee': 0.537},
    'ma200':    {'strat': 0.633, 'ee': 0.536},
    'tsmom12':  {'strat': 0.638, 'ee': 0.538},
}

print(f"{'variant':8s}  {'max_strat':>10} {'max_ee':>8} {'max_Δ':>7}  "
      f"{'cmn_strat':>10} {'cmn_ee':>8} {'cmn_Δ':>7}  Δ변화")
print("-"*75)
for name in NAMES:
    ms = m3_max[name]['strat'];  me = m3_max[name]['ee']
    cs = common_meta[name]['m_strat']['sharpe']
    ce = common_meta[name]['m_ee']['sharpe']
    md = ms - me;  cd = cs - ce
    chg = cd - md
    tag = "↓기간탓" if chg < -0.005 else ("↑" if chg > 0.005 else "≈")
    print(f"{name:8s}  {ms:10.3f} {me:8.3f} {md:+7.3f}  "
          f"{cs:10.3f} {ce:8.3f} {cd:+7.3f}  {chg:+.3f} {tag}")
print()
print("※ cmn_Δ가 진짜 정보원 간 비교값 (공통기간 기준)")
print("  추세: 1988~1990 초기 강세 구간 제외 효과로 우위 폭 소폭 축소 — 구조적 변화 아님")


## 3. 세 상관행렬

In [ ]:
common_idx = signals_common['realized'].index
for n in NAMES:
    common_idx = common_idx.intersection(returns_strat[n].index)
bnh_ref = returns_bnh['vix'].reindex(common_idx)   # 공통 B&H (모든 변형 동일)

df_sig = pd.DataFrame({n: signals_common[n].reindex(common_idx) for n in NAMES})
df_ret = pd.DataFrame({n: returns_strat[n].reindex(common_idx)  for n in NAMES})
df_act = pd.DataFrame({n: (returns_strat[n].reindex(common_idx) - bnh_ref) for n in NAMES})

corr_sig = df_sig.corr()
corr_ret = df_ret.corr()
corr_act = df_act.corr()

corr_sig.round(4).to_csv('results/independence_signal_corr.csv')
corr_ret.round(4).to_csv('results/independence_return_corr.csv')
corr_act.round(4).to_csv('results/independence_active_corr.csv')

def show_corr(label, corr):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    hdr = f"{'':10s}" + "".join(f"{n:>9s}" for n in NAMES)
    print(hdr); print("-"*60)
    for r in NAMES:
        row = f"{r:10s}" + "".join(f"{corr.loc[r,c]:9.3f}" for c in NAMES)
        print(row)

show_corr("(i) 신호 w_t 상관 — 타이밍 겹침 직접 측정", corr_sig)
show_corr("(ii) 원시 net 수익 상관 — 공통 주식베타로 0.7~1.0 부풀려짐", corr_ret)
show_corr("(iii) 능동수익(전략-B&H) 상관 — Gate 1 판정 기준", corr_act)
print()
print("저장: independence_signal_corr.csv / _return_corr.csv / _active_corr.csv")


## 4. family 내/간 상관 구조 대비

In [ ]:
def classify(rho):
    ar = abs(rho)
    if ar < CORR_LOW:  return "LOW"
    if ar > CORR_HIGH: return "HIGH"
    return "gray"

print("=== family 내부 쌍 (능동수익 iii) ===")
for a in NAMES:
    for b in NAMES:
        if b <= a: continue
        if FAMILIES[a] == FAMILIES[b]:
            r_act = float(corr_act.loc[a,b])
            r_sig = float(corr_sig.loc[a,b])
            print(f"  {a:8s} ↔ {b:8s}:  ρ_act={r_act:+.3f} [{classify(r_act)}]  ρ_sig={r_sig:+.3f}")

print()
print("=== family 간 쌍 (정보원 독립성 핵심) ===")
for a in NAMES:
    for b in NAMES:
        if b <= a: continue
        if FAMILIES[a] != FAMILIES[b]:
            r_act = float(corr_act.loc[a,b])
            r_sig = float(corr_sig.loc[a,b])
            c_act = classify(r_act); c_sig = classify(r_sig)
            print(f"  {a:8s} ↔ {b:8s}:  ρ_act={r_act:+.3f} [{c_act}]  ρ_sig={r_sig:+.3f} [{c_sig}]")

print()
print("관찰: 능동수익 상관(iii)이 모든 쌍에서 HIGH(>0.50)인 이유")
print("  능동수익 = 전략수익 - B&H ≈ -(1-w_t)×(r_eq-rf)")
print("  주식 초과수익(r_eq-rf)의 분산이 (1-w_t) 변동보다 훨씬 크므로")
print("  세 정보원이 모두 crisis에 노출 축소 → 양(+) 능동수익 동시 발생 → 고상관.")
print("  신호 w_t 상관(i)이 타이밍 독립성을 더 직접 측정함.")
print("  그러나 Gate 1은 사전 등록 임계(iii 기준) 그대로 적용한다.")


## 5. Gate 1 포함/제외 판정

In [ ]:
# --- 기준1: 공통기간 Sharpe/Sortino/Calmar 중 2개 이상 > ee ---
crit1 = {}
print("=== 기준1: 공통기간 위험조정 vs ee ===")
print(f"{'variant':8s}  {'Sh_s':>7} {'Sh_e':>7} {'Sh_Δ':>6}  "
      f"{'So_s':>7} {'So_e':>7} {'So_Δ':>6}  "
      f"{'Cal_s':>7} {'Cal_e':>7} {'Cal_Δ':>6}  기준1")
print("-"*90)
for name in NAMES:
    ms = common_meta[name]['m_strat']; me = common_meta[name]['m_ee']
    sh_s,sh_e = ms['sharpe'],me['sharpe']
    so_s,so_e = ms['sortino'],me['sortino']
    ca_s,ca_e = ms['calmar'],me['calmar']
    beats = sum([sh_s>sh_e, so_s>so_e, ca_s>ca_e])
    crit1[name] = (beats >= 2)
    print(f"{name:8s}  {sh_s:7.3f} {sh_e:7.3f} {sh_s-sh_e:+6.3f}  "
          f"{so_s:7.3f} {so_e:7.3f} {so_s-so_e:+6.3f}  "
          f"{ca_s:7.3f} {ca_e:7.3f} {ca_s-ca_e:+6.3f}  "
          f"{'PASS('+str(beats)+'/3)' if crit1[name] else 'FAIL('+str(beats)+'/3)'}")

# --- 기준2: 능동수익 상관(iii) — 정보원 간 쌍 ---
print()
print(f"=== 기준2: 능동수익 상관(iii) family 간  [LOW<{CORR_LOW}, HIGH>{CORR_HIGH}] ===")
credit_cross_rhos = []
for a in NAMES:
    for b in NAMES:
        if b <= a: continue
        if FAMILIES[a] == FAMILIES[b]: continue
        rho = float(corr_act.loc[a,b])
        c   = classify(rho)
        if 'credit' in (a,b):
            credit_cross_rhos.append(rho)
        print(f"  {a:8s} ↔ {b:8s}: {rho:+.3f} [{c}]")

credit_all_high = all(abs(r) > CORR_HIGH for r in credit_cross_rhos)
SOURCE_VARIANTS = {
    'volatility': ['vix','realized','blend'],
    'credit':     ['credit'],
    'trend':      ['ma200','tsmom12'],
}

print()
print("="*65)
print("■ Gate 1 최종 판정 (사전 등록 임계 그대로 적용)")
print(f"  능동수익 임계: |ρ|<{CORR_LOW}=저상관, |ρ|>{CORR_HIGH}=고상관")
print("="*65)
include_list = []
for src, variants in SOURCE_VARIANTS.items():
    c1 = any(crit1[v] for v in variants)
    if src == 'credit':
        if c1:
            verdict = "INCLUDE (기준1)"; include_list.append(src)
            rsn = f"Sharpe Δ={common_meta['credit']['m_strat']['sharpe']-common_meta['credit']['m_ee']['sharpe']:+.3f}"
        elif not credit_all_high:
            verdict = "INCLUDE (기준2)"; include_list.append(src)
            rsn = "저상관 쌍 존재 → 분산이득 기대"
        else:
            verdict = "★ EXCLUDE"
            rlo = min(abs(r) for r in credit_cross_rhos)
            rhi = max(abs(r) for r in credit_cross_rhos)
            rsn = (f"기준1 FAIL (Sharpe {common_meta['credit']['m_strat']['sharpe']:.3f} < "
                   f"ee {common_meta['credit']['m_ee']['sharpe']:.3f})
"
                   f"              기준2 FAIL (능동수익 상관 {rlo:.3f}~{rhi:.3f}, 모두 > {CORR_HIGH})")
    else:
        best = max(variants, key=lambda v: common_meta[v]['m_strat']['sharpe'])
        if c1:
            verdict = "INCLUDE (기준1)"; include_list.append(src)
            rsn = f"{best} Sharpe {common_meta[best]['m_strat']['sharpe']:.3f} > ee {common_meta[best]['m_ee']['sharpe']:.3f}"
        else:
            verdict = "EXCLUDE (기준1 미충족)"
            rsn = "공통기간 모든 변형이 ee를 2/3 지표로 이기지 못함"
    print(f"\n  [{src.upper():12s}]  {verdict}")
    print(f"   근거: {rsn}")

print()
print("─"*65)
print(f"M5 결합 포함 명단: {' + '.join(s.upper() for s in include_list)}")
if 'credit' not in include_list:
    print("CREDIT 제외 (정직한 음의 결론 — 임계 조정·변호 없음)")
print("─"*65)


## 6. 상관 히트맵

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles = [
    '(i) 신호 w_t 상관',
    '(ii) 원시 수익 상관',
    '(iii) 능동수익 상관 (Gate 1 판정 기준)'
]
for ax, corr, title in zip(axes, [corr_sig, corr_ret, corr_act], titles):
    vals = corr.values
    im = ax.imshow(vals, vmin=-1, vmax=1, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(range(6)); ax.set_yticks(range(6))
    ax.set_xticklabels(NAMES, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(NAMES, fontsize=8)
    for i in range(6):
        for j in range(6):
            v = float(corr.iloc[i, j])
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=7, color='black' if abs(v) < 0.7 else 'white')
    ax.set_title(title, fontsize=8, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle(
    f'M4 독립성 분석 — 공통기간 {COMMON_START.date()}~  '
    f'Gate 1: |ρ_active|<{CORR_LOW}(저)·>{CORR_HIGH}(고)',
    fontsize=9, y=1.01
)
plt.tight_layout()
plt.savefig('results/independence_corr_heatmap.png', dpi=120, bbox_inches='tight')
plt.close()
print("저장: results/independence_corr_heatmap.png")


## 7. Gate 1 해석 노트

### 능동수익 상관이 모두 HIGH인 이유
능동수익 = 전략수익 − B&H ≈ −(1−w_t)×(r_eq − rf).  
주식 초과수익의 변동성이 (1−w_t) 변동보다 훨씬 크기 때문에,  
세 정보원이 서로 다른 신호를 쓰더라도 위기 시 노출 축소로 동시에 양(+) 능동수익을 기록 → 고상관.  
이는 공통 주식베타 노출 구조의 수학적 귀결이지, 신호가 같은 것을 재는 게 아니다.

### 신용 제외 판정
- **기준1:** 공통기간 Sharpe 0.528 < ee 0.531 (Δ = −0.003). 3개 지표 중 1개(Calmar만)만 우위.
- **기준2:** 능동수익 상관 0.720~0.886 — 모두 HIGH(> 0.50). 사전 등록 임계 적용.
- **정직한 음의 결론:** BAA10Y 단독으로는 동일노출 정적 배분을 이기지 못하고,  
  능동수익 고상관으로 분산이득도 기대하기 어렵다. 사전 등록 기준에 따라 제외.  
  (신호 w_t 상관에서 credit-tsmom12=0.288(LOW)이 눈에 띠지만,  
   Gate 1 판정 임계는 능동수익(iii) 기준으로 사전 등록됨 — 변호 없음.)

### M5 진입 명단: VOLATILITY + TREND
- 가족 내 어느 변형을 결합 대표로 쓸지는 M5에서 결정.
- 두 정보원의 능동수익 상관 0.779~0.885도 HIGH — M5 결합의 분산이득이 제한적일 수 있음을 미리 인식.
